# RivalRadar Agentic Pipeline (Agent 1-4)

Single notebook pipeline:
- Agent 1: scrape + parse competitor pages
- Agent 2: OpenAI vulnerability analysis
- Agent 3: OpenAI pricing-change prediction
- Agent 4: OpenAI action recommendation

Goal: run one end-to-end pipeline in one notebook.

## 1. Imports, OpenAI Key, and Shared Configuration

In [82]:
import asyncio
import json
import os
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from openai import OpenAI

from competitor_targets import COMPETITOR_TARGETS
from scrapers.output_parser import parse_pricing_page, save_structured

# Crawlee is optional for live scraping; fallback mode can still run without it.
try:
    from scrapers.crawlee_scraper import CrawleeScraper
    CRAWLEE_AVAILABLE = True
except ModuleNotFoundError:
    CrawleeScraper = None
    CRAWLEE_AVAILABLE = False

OUTPUT_DIR = Path("scraper_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── API Key ───────────────────────────────────────────────────────────────────
# Load from .env.example in the same directory as this notebook.
def _load_env_file(path: Path) -> dict:
    env = {}
    if path.exists():
        for line in path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, _, v = line.partition("=")
                env[k.strip()] = v.strip()
    return env

_env = _load_env_file(Path(".env.example"))
OPENAI_API_KEY = _env.get("OPENAI_API_KEY", "").strip()

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found or empty in .env.example")
print("✓ API key loaded from .env.example")
# ─────────────────────────────────────────────────────────────────────────────

OPENAI_MODEL = _env.get("OPENAI_MODEL", "gpt-4o-mini")

RUN_LIVE_SCRAPE = CRAWLEE_AVAILABLE
VERBOSE = True
MAX_RETRIES = 1

print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Competitors configured: {len(COMPETITOR_TARGETS)}")
print(f"Crawlee available: {CRAWLEE_AVAILABLE}")
print(f"Live scrape mode: {RUN_LIVE_SCRAPE}")
print(f"OpenAI model: {OPENAI_MODEL}")

if not CRAWLEE_AVAILABLE:
    print("Install missing deps in your venv: pip install -r requirements.txt")


def get_openai_client() -> OpenAI:
    return OpenAI(api_key=OPENAI_API_KEY)


def chat_json(client: OpenAI, system_prompt: str, user_prompt: str, temperature: float = 0.2) -> dict[str, Any]:
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        response_format={"type": "json_object"},
    )
    content = response.choices[0].message.content
    if not content:
        raise RuntimeError("OpenAI returned empty content")
    return json.loads(content)

✓ API key loaded from .env.example
Output dir: /Users/hrithikkannankrishnan/Desktop/Innovation Challenge /DBA5115-rivalradar/rivalradar/scraper_output
Competitors configured: 5
Crawlee available: True
Live scrape mode: True
OpenAI model: gpt-4o-mini


## 2. Define First Agent Interface and Implementation

In [83]:
class Agent1Collector:
    """Collect competitor pricing pages and emit structured competitor profiles."""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.scraper = CrawleeScraper() if CRAWLEE_AVAILABLE else None

    def build_targets(self) -> list[dict[str, str]]:
        return [
            {
                "url": target["pricing_url"],
                "page_type": "pricing",
                "name": target["name"],
            }
            for target in COMPETITOR_TARGETS
        ]

    async def collect(self) -> dict[str, Any]:
        if self.scraper is None:
            raise ModuleNotFoundError(
                "crawlee is not installed. Install deps with: pip install -r requirements.txt"
            )

        targets = self.build_targets()
        scrape_results = await self.scraper.scrape_pages(targets)

        profiles: list[dict[str, Any]] = []
        for result in scrape_results:
            profile = parse_pricing_page(
                raw_text=result.get("raw_text", ""),
                company=result.get("name", "Unknown"),
                url=result.get("url", ""),
                scraped_at=result.get("scraped_at", ""),
            )
            save_structured(profile, self.output_dir)
            profiles.append(profile)

        return {
            "agent": "agent1_collector",
            "target_count": len(targets),
            "scrape_result_count": len(scrape_results),
            "structured_profiles": profiles,
        }


def load_existing_structured_profiles(output_dir: Path) -> list[dict[str, Any]]:
    profiles: list[dict[str, Any]] = []
    for path in sorted(output_dir.glob("*_structured.json")):
        with open(path, encoding="utf-8") as f:
            profiles.append(json.load(f))
    return profiles

## 3. Define Agent 2 (OpenAI Vulnerability Analysis)

In [84]:
@dataclass
class ScoreComponent:
    name: str
    weight: float
    score: float
    weighted_score: float
    reasoning: str


@dataclass
class VulnerabilityResult:
    company: str
    vulnerability_score: float
    risk_level: str
    confidence: float
    reasoning_summary: str
    detailed_reasoning: list[str]
    decision_trace: list[str]
    component_breakdown: list[ScoreComponent]
    signals: dict[str, int]
    metrics: dict[str, Any]
    scraped_at: str
    peer_rank: int | None
    peer_percentile: float | None


class Agent2CompetitiveAnalyzer:
    """Analyze competitor profiles with OpenAI and return explainable vulnerability results."""

    component_weights = {
        "pricing_pressure": 0.28,
        "segment_coverage": 0.22,
        "feature_depth": 0.18,
        "strategic_signals": 0.20,
        "business_model_pressure": 0.12,
    }

    def __init__(self, output_dir: Path, client: OpenAI):
        self.output_dir = output_dir
        self.client = client

    def _analyze_one(self, profile: dict[str, Any]) -> VulnerabilityResult:
        prompt = (
            "You are a senior competitive intelligence analyst. Return strict JSON only. "
            "Analyze this competitor profile and produce vulnerability analysis.\n"
            "Required keys: vulnerability_score, confidence, risk_level, reasoning_summary, "
            "detailed_reasoning, decision_trace, component_breakdown, signals, metrics.\n"
            "component_breakdown must include these names exactly: pricing_pressure, segment_coverage, "
            "feature_depth, strategic_signals, business_model_pressure.\n"
            "signals must include ai_momentum, enterprise_readiness, integration_strength as integers.\n"
            f"Profile: {json.dumps(profile, ensure_ascii=False)}"
        )
        payload = chat_json(
            self.client,
            "Return only valid JSON, no markdown.",
            prompt,
            temperature=0.1,
        )

        raw_component_names = [
            item.get("name", "")
            for item in payload.get("component_breakdown", [])
            if isinstance(item, dict)
        ]
        if VERBOSE:
            print(f"[Agent2] Raw component names from LLM: {raw_component_names}")

        components_by_name = {
            item.get("name", "").lower().strip().replace(" ", "_"): item
            for item in payload.get("component_breakdown", [])
            if isinstance(item, dict)
        }

        components: list[ScoreComponent] = []
        for name, weight in self.component_weights.items():
            raw = components_by_name.get(name, {})
            score = max(0.0, min(1.0, float(raw.get("score", 0.0))))
            components.append(
                ScoreComponent(
                    name=name,
                    weight=weight,
                    score=round(score, 3),
                    weighted_score=round(score * weight, 4),
                    reasoning=str(raw.get("reasoning", "No reasoning provided.")).strip(),
                )
            )

        vulnerability_score = max(0.0, min(1.0, float(payload.get("vulnerability_score", sum(c.weighted_score for c in components)))))
        confidence = max(0.0, min(1.0, float(payload.get("confidence", 0.6))))
        risk_level = str(payload.get("risk_level", "medium")).lower().strip()
        if risk_level not in {"low", "medium", "high", "critical"}:
            risk_level = "high" if vulnerability_score >= 0.65 else ("medium" if vulnerability_score >= 0.45 else "low")

        signals_raw = payload.get("signals", {}) if isinstance(payload.get("signals"), dict) else {}
        signals = {
            "ai_momentum": int(signals_raw.get("ai_momentum", 0)),
            "enterprise_readiness": int(signals_raw.get("enterprise_readiness", 0)),
            "integration_strength": int(signals_raw.get("integration_strength", 0)),
        }

        metrics_raw = payload.get("metrics", {}) if isinstance(payload.get("metrics"), dict) else {}
        metrics = {
            "plan_count": int(metrics_raw.get("plan_count", len(profile.get("plans", [])))),
            "price_range": metrics_raw.get("price_range", profile.get("pricing_summary", {}).get("price_range", "N/A")),
            "raw_char_count": int(metrics_raw.get("raw_char_count", profile.get("raw_char_count", 0))),
            "avg_features_per_plan": float(metrics_raw.get("avg_features_per_plan", 0.0)),
            "unique_feature_count": int(metrics_raw.get("unique_feature_count", 0)),
            "numeric_price_count": int(metrics_raw.get("numeric_price_count", 0)),
        }

        detailed_reasoning = payload.get("detailed_reasoning", [])
        if not isinstance(detailed_reasoning, list):
            detailed_reasoning = [str(detailed_reasoning)]

        decision_trace = payload.get("decision_trace", [])
        if not isinstance(decision_trace, list):
            decision_trace = [str(decision_trace)]

        return VulnerabilityResult(
            company=profile.get("company", "Unknown"),
            vulnerability_score=round(vulnerability_score, 3),
            risk_level=risk_level,
            confidence=round(confidence, 3),
            reasoning_summary=str(payload.get("reasoning_summary", "No summary provided.")).strip(),
            detailed_reasoning=[str(x) for x in detailed_reasoning],
            decision_trace=[str(x) for x in decision_trace],
            component_breakdown=components,
            signals=signals,
            metrics=metrics,
            scraped_at=profile.get("scraped_at", datetime.now(timezone.utc).isoformat()),
            peer_rank=None,
            peer_percentile=None,
        )

    def analyze(self, agent1_output: dict[str, Any]) -> dict[str, Any]:
        profiles = agent1_output["structured_profiles"]
        results = [self._analyze_one(profile) for profile in profiles]
        results.sort(key=lambda r: r.vulnerability_score, reverse=True)

        total = len(results)
        for idx, result in enumerate(results, start=1):
            result.peer_rank = idx
            result.peer_percentile = 1.0 if total == 1 else round(1 - ((idx - 1) / (total - 1)), 3)
            result.reasoning_summary = (
                f"Ranked #{result.peer_rank}/{total} (peer_percentile={result.peer_percentile:.3f}). "
                f"{result.reasoning_summary}"
            )

        report_path = self.output_dir / "vulnerability_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "generated_at": datetime.now(timezone.utc).isoformat(),
                    "model": "openai_vulnerability_v1",
                    "results": [asdict(r) for r in results],
                },
                f,
                indent=2,
                ensure_ascii=False,
            )

        return {
            "agent": "agent2_analyzer",
            "vulnerability_results": results,
            "report_path": str(report_path),
            "analyzed_count": len(results),
        }

## 4. Define Agent 3 (Pricing Predictor)

In [ ]:
@dataclass
class PricingPrediction:
    company: str
    change_probability: float
    predicted_timeline: str
    risk_level: str
    reasoning_summary: str
    drivers: dict[str, float]
    decision_trace: list[str]
    evidence: dict[str, Any]
    generated_at: str


class Agent3PricingPredictor:
    """Predict competitor pricing-change likelihood and timeline using OpenAI."""

    def __init__(self, output_dir: Path, client: OpenAI):
        self.output_dir = output_dir
        self.client = client

    def _predict_one(self, vuln: VulnerabilityResult, profile: dict[str, Any]) -> PricingPrediction:
        context = {
            "company": vuln.company,
            "vulnerability_score": vuln.vulnerability_score,
            "risk_level": vuln.risk_level,
            "confidence": vuln.confidence,
            "signals": vuln.signals,
            "metrics": vuln.metrics,
            "profile": profile,
        }
        prompt = (
            "Predict pricing change likelihood and timeline. Return strict JSON only.\n"
            "Required keys: change_probability, predicted_timeline, risk_level, reasoning_summary, drivers, decision_trace, evidence.\n"
            "risk_level must be low/medium/high.\n"
            "predicted_timeline must be one of: 0-30 days, 1-2 months, 2-3 months, 3+ months.\n"
            "drivers must include: vulnerability_pressure, strategic_signal_intensity, plan_complexity, "
            "portfolio_competition_intensity, pricing_plan_volatility, analysis_confidence with [0,1] floats.\n"
            f"Context: {json.dumps(context, ensure_ascii=False)}"
        )

        payload = chat_json(self.client, "Return only valid JSON.", prompt, temperature=0.15)
        probability = max(0.0, min(1.0, float(payload.get("change_probability", 0.0))))
        risk = str(payload.get("risk_level", "low")).lower().strip()
        if risk not in {"low", "medium", "high"}:
            risk = "high" if probability >= 0.75 else ("medium" if probability >= 0.5 else "low")

        timeline = str(payload.get("predicted_timeline", "3+ months")).strip()
        if timeline not in {"0-30 days", "1-2 months", "2-3 months", "3+ months"}:
            timeline = "0-30 days" if probability >= 0.8 else ("1-2 months" if probability >= 0.65 else ("2-3 months" if probability >= 0.5 else "3+ months"))

        raw_drivers = payload.get("drivers", {}) if isinstance(payload.get("drivers"), dict) else {}
        drivers = {
            "vulnerability_pressure": round(max(0.0, min(1.0, float(raw_drivers.get("vulnerability_pressure", vuln.vulnerability_score)))), 3),
            "strategic_signal_intensity": round(max(0.0, min(1.0, float(raw_drivers.get("strategic_signal_intensity", 0.0)))), 3),
            "plan_complexity": round(max(0.0, min(1.0, float(raw_drivers.get("plan_complexity", 0.0)))), 3),
            "portfolio_competition_intensity": round(max(0.0, min(1.0, float(raw_drivers.get("portfolio_competition_intensity", 0.0)))), 3),
            "pricing_plan_volatility": round(max(0.0, min(1.0, float(raw_drivers.get("pricing_plan_volatility", 0.0)))), 3),
            "analysis_confidence": round(max(0.0, min(1.0, float(raw_drivers.get("analysis_confidence", vuln.confidence)))), 3),
        }

        decision_trace = payload.get("decision_trace", [])
        if not isinstance(decision_trace, list):
            decision_trace = [str(decision_trace)]

        evidence = payload.get("evidence", {}) if isinstance(payload.get("evidence"), dict) else {}
        evidence.setdefault("price_range", vuln.metrics.get("price_range", "N/A"))
        evidence.setdefault("peer_rank", vuln.peer_rank)

        return PricingPrediction(
            company=vuln.company,
            change_probability=round(probability, 3),
            predicted_timeline=timeline,
            risk_level=risk,
            reasoning_summary=str(payload.get("reasoning_summary", "No summary provided.")).strip(),
            drivers=drivers,
            decision_trace=[str(x) for x in decision_trace],
            evidence=evidence,
            generated_at=datetime.now(timezone.utc).isoformat(),
        )

    def predict(self, agent1_output: dict[str, Any], agent2_output: dict[str, Any]) -> dict[str, Any]:
        profiles = {p.get("company", "Unknown"): p for p in agent1_output["structured_profiles"]}
        vulnerability_results = agent2_output["vulnerability_results"]

        predictions = [
            self._predict_one(vuln, profiles.get(vuln.company, {}))
            for vuln in vulnerability_results
        ]
        predictions.sort(key=lambda p: p.change_probability, reverse=True)

        report_path = self.output_dir / "pricing_predictions_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "generated_at": datetime.now(timezone.utc).isoformat(),
                    "model": "openai_pricing_prediction_v1",
                    "results": [asdict(p) for p in predictions],
                },
                f,
                indent=2,
                ensure_ascii=False,
            )

        return {
            "agent": "agent3_pricing",
            "pricing_predictions": predictions,
            "report_path": str(report_path),
            "prediction_count": len(predictions),
        }

## 5. Define Agent 4 (Action Planner)

In [ ]:
@dataclass
class ActionRecommendation:
    company: str
    priority: str
    recommendation_type: str
    owner: str
    due_window: str
    action_title: str
    action_detail: str
    rationale: str
    evidence: dict[str, Any]
    decision_trace: list[str]
    generated_at: str


class Agent4ActionPlanner:
    """Generate VC-facing action recommendations from Agent 2 + Agent 3 with OpenAI."""

    def __init__(self, output_dir: Path, client: OpenAI):
        self.output_dir = output_dir
        self.client = client

    def _recommend_one(self, vuln: VulnerabilityResult, pred: PricingPrediction) -> ActionRecommendation:
        context = {
            "company": vuln.company,
            "vulnerability_score": vuln.vulnerability_score,
            "vulnerability_risk": vuln.risk_level,
            "vulnerability_summary": vuln.reasoning_summary,
            "pricing_change_probability": pred.change_probability,
            "pricing_timeline": pred.predicted_timeline,
            "pricing_risk": pred.risk_level,
            "pricing_summary": pred.reasoning_summary,
        }
        prompt = (
            "Generate one VC action recommendation. Return strict JSON only.\n"
            "Required keys: priority, recommendation_type, owner, due_window, action_title, action_detail, rationale, evidence, decision_trace.\n"
            "priority must be one of: P0, P1, P2, P3.\n"
            "recommendation_type must be one of: pricing_response, product_response, gtm_response, monitor_only.\n"
            f"Context: {json.dumps(context, ensure_ascii=False)}"
        )

        payload = chat_json(self.client, "Return only valid JSON.", prompt, temperature=0.2)
        priority = str(payload.get("priority", "P2")).upper().strip()
        if priority not in {"P0", "P1", "P2", "P3"}:
            priority = "P2"

        rec_type = str(payload.get("recommendation_type", "monitor_only")).lower().strip()
        if rec_type not in {"pricing_response", "product_response", "gtm_response", "monitor_only"}:
            rec_type = "monitor_only"

        decision_trace = payload.get("decision_trace", [])
        if not isinstance(decision_trace, list):
            decision_trace = [str(decision_trace)]

        evidence = payload.get("evidence", {}) if isinstance(payload.get("evidence"), dict) else {}
        evidence.setdefault("vulnerability_score", vuln.vulnerability_score)
        evidence.setdefault("pricing_change_probability", pred.change_probability)

        return ActionRecommendation(
            company=vuln.company,
            priority=priority,
            recommendation_type=rec_type,
            owner=str(payload.get("owner", "Portfolio Operations Lead")).strip(),
            due_window=str(payload.get("due_window", "14 days")).strip(),
            action_title=str(payload.get("action_title", f"Monitor {vuln.company}")).strip(),
            action_detail=str(payload.get("action_detail", "No detail provided.")).strip(),
            rationale=str(payload.get("rationale", "No rationale provided.")).strip(),
            evidence=evidence,
            decision_trace=[str(x) for x in decision_trace],
            generated_at=datetime.now(timezone.utc).isoformat(),
        )

    def recommend(self, agent2_output: dict[str, Any], agent3_output: dict[str, Any]) -> dict[str, Any]:
        vulnerability_results = agent2_output["vulnerability_results"]
        predictions = {p.company: p for p in agent3_output["pricing_predictions"]}

        recommendations: list[ActionRecommendation] = []
        for vuln in vulnerability_results:
            pred = predictions.get(vuln.company)
            if pred is None:
                continue
            recommendations.append(self._recommend_one(vuln, pred))

        priority_order = {"P0": 0, "P1": 1, "P2": 2, "P3": 3}
        recommendations.sort(key=lambda r: (priority_order.get(r.priority, 9), r.company))

        report_path = self.output_dir / "action_recommendations_report.json"
        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "generated_at": datetime.now(timezone.utc).isoformat(),
                    "model": "openai_action_recommendation_v1",
                    "results": [asdict(r) for r in recommendations],
                },
                f,
                indent=2,
                ensure_ascii=False,
            )

        return {
            "agent": "agent4_actions",
            "action_recommendations": recommendations,
            "report_path": str(report_path),
            "recommendation_count": len(recommendations),
        }

## 6. Compose Multi-Agent Pipeline (Agents 1-4)

In [86]:
async def run_agent_pipeline(run_live_scrape: bool = True) -> dict[str, Any]:
    start = time.perf_counter()

    client = get_openai_client()

    agent1 = Agent1Collector(OUTPUT_DIR)
    agent2 = Agent2CompetitiveAnalyzer(OUTPUT_DIR, client)
    agent3 = Agent3PricingPredictor(OUTPUT_DIR, client)
    agent4 = Agent4ActionPlanner(OUTPUT_DIR, client)

    agent1_output: dict[str, Any] | None = None

    if run_live_scrape:
        for attempt in range(MAX_RETRIES + 1):
            try:
                if VERBOSE:
                    print(f"[Agent1] Live scrape attempt {attempt + 1}/{MAX_RETRIES + 1}")
                agent1_output = await agent1.collect()
                if VERBOSE:
                    print(f"[Agent1] Scraped {len(agent1_output.get('structured_profiles', []))} profiles.")
                break
            except Exception as exc:
                print(f"[Agent1] Attempt failed: {exc}")
                if attempt == MAX_RETRIES:
                    print("[Agent1] Falling back to existing structured JSON files.")

    # Fall back if scrape was skipped, failed, or returned no profiles
    if agent1_output is None or len(agent1_output.get("structured_profiles", [])) == 0:
        profiles = load_existing_structured_profiles(OUTPUT_DIR)
        if VERBOSE:
            print(f"[Agent1] Loaded {len(profiles)} existing structured profiles from disk.")
        agent1_output = {
            "agent": "agent1_collector_fallback",
            "target_count": len(COMPETITOR_TARGETS),
            "scrape_result_count": 0,
            "structured_profiles": profiles,
        }

    agent2_output = agent2.analyze(agent1_output)
    agent3_output = agent3.predict(agent1_output, agent2_output)
    agent4_output = agent4.recommend(agent2_output, agent3_output)

    elapsed = time.perf_counter() - start
    if VERBOSE:
        print(f"[Pipeline] Completed in {elapsed:.2f}s")

    return {
        "elapsed_seconds": elapsed,
        "agent1_output": agent1_output,
        "agent2_output": agent2_output,
        "agent3_output": agent3_output,
        "agent4_output": agent4_output,
    }

## 7. Run End-to-End Example (Agents 1-4)

In [87]:
pipeline_result = await run_agent_pipeline(run_live_scrape=RUN_LIVE_SCRAPE)

print("Agent 1 output keys:", list(pipeline_result["agent1_output"].keys()))
print("Agent 2 output keys:", list(pipeline_result["agent2_output"].keys()))
print("Agent 3 output keys:", list(pipeline_result["agent3_output"].keys()))
print("Agent 4 output keys:", list(pipeline_result["agent4_output"].keys()))

vulnerability_results = pipeline_result["agent2_output"]["vulnerability_results"]
pricing_predictions = pipeline_result["agent3_output"]["pricing_predictions"]
action_recommendations = pipeline_result["agent4_output"]["action_recommendations"]

print("Agent 2 report:", pipeline_result["agent2_output"]["report_path"])
print("Agent 3 report:", pipeline_result["agent3_output"]["report_path"])
print("Agent 4 report:", pipeline_result["agent4_output"]["report_path"])

[scrapers.crawlee_scraper] INFO  No APIFY_TOKEN found — running without proxy.  Set APIFY_TOKEN in .env for residential proxy rotation.
[crawlee.crawlers._playwright._playwright_crawler] INFO  Current request statistics:
┌───────────────────────────────┬──────┐
│ requests_finished             │ 0    │
│ requests_failed               │ 0    │
│ retry_histogram               │ [0]  │
│ request_avg_failed_duration   │ None │
│ request_avg_finished_duration │ None │
│ requests_finished_per_minute  │ 0    │
│ requests_failed_per_minute    │ 0    │
│ request_total_duration        │ 0s   │
│ requests_total                │ 0    │
│ crawler_runtime               │ 0s   │
└───────────────────────────────┴──────┘
[crawlee.crawlers._playwright._playwright_crawler] INFO  Crawled 0/5 pages, 0 failed requests, desired concurrency 3.


[Agent1] Live scrape attempt 1/2


[crawlee._autoscaling.autoscaled_pool] INFO  current_concurrency = 0; desired_concurrency = 3; cpu = 0.0; mem = 0.0; event_loop = 0.0; client_info = 0.0
[crawlee._autoscaling.autoscaled_pool] INFO  Waiting for remaining tasks to finish
[crawlee.crawlers._playwright._playwright_crawler] INFO  Final request statistics:
┌───────────────────────────────┬─────────┐
│ requests_finished             │ 0       │
│ requests_failed               │ 0       │
│ retry_histogram               │ [0]     │
│ request_avg_failed_duration   │ None    │
│ request_avg_finished_duration │ None    │
│ requests_finished_per_minute  │ 0       │
│ requests_failed_per_minute    │ 0       │
│ request_total_duration        │ 0s      │
│ requests_total                │ 0       │
│ crawler_runtime               │ 330.9ms │
└───────────────────────────────┴─────────┘


[Agent1] Scraped 0 profiles.
[Agent1] Loaded 5 existing structured profiles from disk.
[Agent2] Raw component names from LLM: []
[Agent2] Raw component names from LLM: []
[Agent2] Raw component names from LLM: []
[Agent2] Raw component names from LLM: []
[Agent2] Raw component names from LLM: []
[Pipeline] Completed in 73.18s
Agent 1 output keys: ['agent', 'target_count', 'scrape_result_count', 'structured_profiles']
Agent 2 output keys: ['agent', 'vulnerability_results', 'report_path', 'analyzed_count']
Agent 3 output keys: ['agent', 'pricing_predictions', 'report_path', 'prediction_count']
Agent 4 output keys: ['agent', 'action_recommendations', 'report_path', 'recommendation_count']
Agent 2 report: scraper_output/vulnerability_report.json
Agent 3 report: scraper_output/pricing_predictions_report.json
Agent 4 report: scraper_output/action_recommendations_report.json


In [88]:
vulnerability_rows = []
for result in vulnerability_results:
    top_component = max(result.component_breakdown, key=lambda c: c.weighted_score)
    vulnerability_rows.append(
        {
            "company": result.company,
            "rank": result.peer_rank,
            "vulnerability_score": result.vulnerability_score,
            "risk_level": result.risk_level,
            "confidence": result.confidence,
            "top_driver": top_component.name,
            "signal_hits": sum(result.signals.values()),
            "reasoning_summary": result.reasoning_summary,
        }
    )

pricing_rows = [
    {
        "company": p.company,
        "change_probability": p.change_probability,
        "pricing_risk": p.risk_level,
        "predicted_timeline": p.predicted_timeline,
        "top_driver": max(p.drivers.items(), key=lambda item: item[1])[0],
        "reasoning_summary": p.reasoning_summary,
    }
    for p in pricing_predictions
]

action_rows = [
    {
        "company": a.company,
        "priority": a.priority,
        "recommendation_type": a.recommendation_type,
        "owner": a.owner,
        "due_window": a.due_window,
        "action_title": a.action_title,
        "rationale": a.rationale,
    }
    for a in action_recommendations
]

vulnerability_df = pd.DataFrame(vulnerability_rows)
if not vulnerability_df.empty:
    vulnerability_df = vulnerability_df.sort_values("rank")

pricing_df = pd.DataFrame(pricing_rows)
if not pricing_df.empty:
    pricing_df = pricing_df.sort_values("change_probability", ascending=False)

actions_df = pd.DataFrame(action_rows)
if not actions_df.empty:
    actions_df = actions_df.sort_values("priority")

pd.set_option("display.max_colwidth", 220)
print("VULNERABILITY SUMMARY")
display(vulnerability_df)
print("PRICING PREDICTION SUMMARY")
display(pricing_df)
print("ACTION RECOMMENDATION SUMMARY")
display(actions_df)

VULNERABILITY SUMMARY


,company,rank,vulnerability_score,risk_level,confidence,top_driver,signal_hits,reasoning_summary
0,ClickUp,1,1.0,medium,0.85,pricing_pressure,17,"Ranked #1/5 (peer_percentile=1.000). ClickUp's pricing strategy and feature offerings create vulnerabilities in competitive positioning, particularly in the enterprise segment and feature depth."
1,HubSpot,2,1.0,medium,0.75,pricing_pressure,18,Ranked #2/5 (peer_percentile=0.750). HubSpot's pricing structure and feature offerings present vulnerabilities due to potential pricing pressure from competitors and limited feature depth in lower-tier plans.
2,Linear,3,1.0,medium,0.80,pricing_pressure,18,Ranked #3/5 (peer_percentile=0.500). Linear's pricing structure and feature offerings present vulnerabilities due to potential pricing pressure and limited feature depth in lower tiers.
3,Notion,4,1.0,medium,0.75,pricing_pressure,15,Ranked #4/5 (peer_percentile=0.250). Notion's pricing structure and feature offerings present vulnerabilities due to potential pricing pressure from competitors and limited segment coverage in enterprise solutions.
4,Stripe,5,1.0,medium,0.85,pricing_pressure,13,"Ranked #5/5 (peer_percentile=0.000). Stripe's transaction-based pricing model creates vulnerability due to potential pricing pressure from competitors offering fixed plans or lower rates, especially in a market with ..."


PRICING PREDICTION SUMMARY


,company,change_probability,pricing_risk,predicted_timeline,top_driver,reasoning_summary
0,ClickUp,0.6,medium,1-2 months,analysis_confidence,"Given the current vulnerability score and the medium risk level, there is a moderate likelihood of pricing changes in the near future. The company is experiencing significant AI momentum which may drive pricing adjus..."
1,HubSpot,0.6,medium,1-2 months,vulnerability_pressure,"Given the medium risk level and the company's current vulnerability score, there is a moderate likelihood of pricing changes within the next couple of months. The presence of multiple plans and the competitive landsc..."
2,Linear,0.6,medium,1-2 months,vulnerability_pressure,"The company has a medium risk level due to its current vulnerability score and market signals indicating a strong interest in AI and enterprise readiness. However, the lack of unique features and low average features..."
3,Notion,0.6,medium,1-2 months,vulnerability_pressure,"The company is experiencing medium risk due to a high vulnerability score and strong signals in AI momentum and enterprise readiness. However, the pricing plans are stable with a low average feature count, indicating..."
4,Stripe,0.6,medium,1-2 months,analysis_confidence,"Given the current medium risk level and high confidence in the analysis, a moderate likelihood of pricing changes is anticipated within the next couple of months due to competitive pressures and strategic signals."


ACTION RECOMMENDATION SUMMARY


,company,priority,recommendation_type,owner,due_window,action_title,rationale
0,ClickUp,P1,pricing_response,Pricing Strategy Team,1-2 months,Evaluate and Adjust Pricing Strategy,"ClickUp's current pricing strategy has been identified as a vulnerability, particularly in the enterprise segment. Adjusting pricing could strengthen market position and capitalize on AI momentum."
1,HubSpot,P1,pricing_response,Pricing Strategy Team,1-2 months,Evaluate and Adjust Pricing Structure,"Given the medium risk level and the company's current vulnerability score, it is crucial to proactively address potential pricing changes to maintain market competitiveness."
2,Linear,P1,pricing_response,Product Management Team,1-2 months,Evaluate and Adjust Pricing Structure,"Given the medium vulnerability score and the potential pricing pressure from competitors, it is crucial to reassess the pricing strategy to maintain market position and customer satisfaction."
3,Notion,P1,pricing_response,Product Management Team,1-2 months,Review and Adjust Pricing Strategy,"Given the medium vulnerability score and the potential pricing pressure from competitors, it is crucial to proactively assess and potentially adjust the pricing strategy to maintain competitiveness in the market."
4,Stripe,P1,pricing_response,Product Strategy Team,1-2 months,Evaluate and Adjust Pricing Strategy,"The current transaction-based pricing model exposes Stripe to significant competitive risks, especially as competitors may offer more attractive fixed pricing plans. Adjusting the pricing strategy could help retain m..."


## 8. Validation and Debug Output (Agents 1-4)

In [89]:
agent1_output = pipeline_result["agent1_output"]
agent2_output = pipeline_result["agent2_output"]
agent3_output = pipeline_result["agent3_output"]
agent4_output = pipeline_result["agent4_output"]

assert "structured_profiles" in agent1_output, "Agent 1 output missing structured_profiles"
assert isinstance(agent1_output["structured_profiles"], list), "structured_profiles must be a list"
assert len(agent1_output["structured_profiles"]) > 0, "No structured profiles available"

assert "vulnerability_results" in agent2_output, "Agent 2 output missing vulnerability_results"
assert isinstance(agent2_output["vulnerability_results"], list), "vulnerability_results must be a list"
assert len(agent2_output["vulnerability_results"]) > 0, "Agent 2 returned no vulnerability results"

assert "pricing_predictions" in agent3_output, "Agent 3 output missing pricing_predictions"
assert isinstance(agent3_output["pricing_predictions"], list), "pricing_predictions must be a list"
assert len(agent3_output["pricing_predictions"]) > 0, "Agent 3 returned no pricing predictions"

assert "action_recommendations" in agent4_output, "Agent 4 output missing action_recommendations"
assert isinstance(agent4_output["action_recommendations"], list), "action_recommendations must be a list"
assert len(agent4_output["action_recommendations"]) > 0, "Agent 4 returned no recommendations"

sample_profile = agent1_output["structured_profiles"][0]
required_profile_keys = {"company", "pricing_summary", "raw_char_count"}
assert required_profile_keys.issubset(sample_profile.keys()), "Profile schema mismatch"

if VERBOSE:
    print("[Validation] Agents 1-4 outputs passed schema checks.")
    print("[Debug] Structured profiles:", len(agent1_output["structured_profiles"]))
    print("[Debug] Vulnerability results:", len(agent2_output["vulnerability_results"]))
    print("[Debug] Pricing predictions:", len(agent3_output["pricing_predictions"]))
    print("[Debug] Action recommendations:", len(agent4_output["action_recommendations"]))
    print("[Debug] Pipeline elapsed seconds:", round(pipeline_result["elapsed_seconds"], 2))

[Validation] Agents 1-4 outputs passed schema checks.
[Debug] Structured profiles: 5
[Debug] Vulnerability results: 5
[Debug] Pricing predictions: 5
[Debug] Action recommendations: 5
[Debug] Pipeline elapsed seconds: 73.18
